# Teorema del Límite Central
**Estadística Bayesiana** | UAEMex Valle de México  
Ximena Quintanar Leal · 2025

---


## 1. Enunciado

Sea $X_1, X_2, \ldots, X_n$ una sucesión de variables aleatorias i.i.d. con media $\mu$ y varianza $\sigma^2 < \infty$.  
Definimos la suma estandarizada:

$$Z_n = \frac{\bar{X}_n - \mu}{\sigma / \sqrt{n}}$$

Entonces, conforme $n \to \infty$:

$$Z_n \xrightarrow{d} \mathcal{N}(0,1)$$

Este resultado es fundamental porque **no requiere conocer la distribución** de las $X_i$, solo que existan su media y varianza.


## 2. Relevancia en el contexto bayesiano

En inferencia bayesiana, el TLC justifica las **aproximaciones normales a los posteriors** cuando el tamaño de muestra es grande (*Bernstein–von Mises theorem*):

$$p(\theta \mid x_1, \ldots, x_n) \approx \mathcal{N}\!\left(\hat{\theta}_{\text{MLE}},\; \frac{1}{n\, \mathcal{I}(\hat{\theta})}\right)$$

donde $\mathcal{I}(\hat{\theta})$ es la información de Fisher evaluada en el MLE.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, expon, uniform

rng = np.random.default_rng(42)

distribuciones = {
    'Uniforme U(0,1)':    lambda n: rng.uniform(0, 1, n),
    'Exponencial λ=1':    lambda n: rng.exponential(1, n),
    'Bernoulli p=0.3':    lambda n: rng.binomial(1, 0.3, n).astype(float),
}

tamanios = [1, 5, 30, 200]
N_sims   = 3000

fig, axes = plt.subplots(len(distribuciones), len(tamanios),
                         figsize=(13, 9), sharex=False)

for row, (nombre, sampler) in enumerate(distribuciones.items()):
    for col, n in enumerate(tamanios):
        medias = np.array([sampler(n).mean() for _ in range(N_sims)])
        mu_est = medias.mean()
        se_est = medias.std()
        z = (medias - mu_est) / se_est

        ax = axes[row][col]
        ax.hist(z, bins=40, density=True, color='steelblue',
                alpha=0.65, edgecolor='white', linewidth=0.4)
        x_line = np.linspace(-4, 4, 200)
        ax.plot(x_line, norm.pdf(x_line), color='tomato', linewidth=1.6)
        ax.set_title(f'n = {n}', fontsize=10)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        if col == 0:
            ax.set_ylabel(nombre, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle('Teorema del Límite Central — convergencia a N(0,1)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 3. Velocidad de convergencia — Desigualdad de Berry-Esseen

La velocidad con que $Z_n$ converge a la normal está acotada por:

$$\sup_x \left| F_n(x) - \Phi(x) \right| \leq \frac{C\, \rho}{\sigma^3 \sqrt{n}}$$

donde $\rho = E[|X - \mu|^3]$ y $C \leq 0.4748$.

**Interpretación:** distribuciones más simétricas y con colas más ligeras convergen más rápido.


In [ ]:
# Comparación QQ-plot para distintos n
from scipy.stats import probplot

fig, axes = plt.subplots(1, 4, figsize=(13, 4))

for ax, n in zip(axes, tamanios):
    medias = np.array([rng.exponential(1, n).mean() for _ in range(N_sims)])
    z = (medias - medias.mean()) / medias.std()
    probplot(z, dist='norm', plot=ax)
    ax.set_title(f'QQ-plot  n={n}', fontsize=10)
    ax.get_lines()[0].set(markersize=2, alpha=0.4, color='steelblue')
    ax.get_lines()[1].set(color='tomato', linewidth=1.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Convergencia a la normal (Exponencial) — QQ plots', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


---
## Referencias
- Billingsley, P. (1995). *Probability and Measure*, 3rd ed. Wiley.
- Gelman, A. et al. (2013). *Bayesian Data Analysis*, 3rd ed. — Cap. 4 (Large-sample theory).
